# Experiment 5: Morphology and POS Tagging

- **Morpheme** = smallest meaningful part: `unhappiness = un + happy + ness`.
- **Inflectional** suffix keeps the word (`walk -> walked`). **Derivational** makes a new word (`happy -> happiness`).
- **POS tagging** labels each word: Penn tags (NN, VBD, JJ...) or Universal tags (NOUN, VERB...).
- Taggers: Default (all NN) < Unigram < Bigram < Perceptron.

In [ ]:
# !pip install nltk spacy
# !python -m spacy download en_core_web_sm

In [ ]:
import nltk, spacy
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tag import DefaultTagger, UnigramTagger, BigramTagger, PerceptronTagger
from nltk.corpus import treebank

for r in ['punkt', 'punkt_tab', 'averaged_perceptron_tagger_eng', 'universal_tagset',
          'wordnet', 'omw-1.4', 'treebank']:
    nltk.download(r, quiet=True)

nlp = spacy.load('en_core_web_sm')
stemmer, lemmatizer = PorterStemmer(), WordNetLemmatizer()

## Simple affix analysis

In [ ]:
PREFIXES = ['inter', 'mis', 'dis', 'pre', 'un', 're']
SUFFIXES = {'ization': 'derivational', 'ness': 'derivational', 'ment': 'derivational',
            'able': 'derivational', 'ly': 'derivational',
            'ing': 'inflectional', 'ed': 'inflectional', 'est': 'inflectional', 's': 'inflectional'}

def affixes(word):
    pre = next((p for p in PREFIXES if word.startswith(p) and len(word) > len(p) + 3), '-')
    suf = next((s for s in SUFFIXES if word.endswith(s) and len(word) > len(s) + 2), '-')
    return pre, suf, SUFFIXES.get(suf, '-')

for w in ['unhappiness', 'rewriting', 'misunderstood', 'players', 'quickly', 'enjoyable', 'fastest']:
    pre, suf, kind = affixes(w)
    print(f'{w:14} prefix={pre:6} suffix={suf:6} {kind:13} stem={stemmer.stem(w):10} lemma={lemmatizer.lemmatize(w, "v")}')

## Morphological features with SpaCy

In [ ]:
for t in nlp('The children were playing happily, but she wrote the longest essays.'):
    if not t.is_punct:
        print(f'{t.text:10} lemma={t.lemma_:8} pos={t.pos_:6} {t.morph}')

## POS tagging: Penn vs Universal

In [ ]:
tokens = word_tokenize('The quick brown fox jumps over the lazy dog.')
print(nltk.pos_tag(tokens))                       # Penn Treebank tags
print(nltk.pos_tag(tokens, tagset='universal'))   # Universal tags

## Ambiguity: same word, different tag

In [ ]:
for sent in ['I can can a can.', 'Please book a table and bring the book.',
             'Turn on the light because the bag is light.']:
    print([(t.text, t.tag_) for t in nlp(sent)])

## Train and compare taggers on the Penn Treebank (80/20 split)

In [ ]:
sents = [[(w, t) for w, t in s if t != '-NONE-'] for s in treebank.tagged_sents()]
cut = int(0.8 * len(sents))
train, test = sents[:cut], sents[cut:]

default = DefaultTagger('NN')                          # tags everything NN
unigram = UnigramTagger(train, backoff=default)        # most common tag per word
bigram = BigramTagger(train, backoff=unigram)          # also looks at previous tag
perceptron = PerceptronTagger(load=False)
perceptron.train(train, nr_iter=5)                     # learns from many features

for name, tagger in [('Default', default), ('Unigram', unigram),
                     ('Bigram', bigram), ('Perceptron', perceptron)]:
    print(f'{name:10} accuracy = {tagger.accuracy(test):.1%}')